# 🌡️ Thermal Network Analysis

Dieses Notebook analysiert die Ergebnisse des thermischen Fernwärmenetzes.

## Features:
- Detaillierte Netzwerk-KPIs
- Temperaturprofile an allen Knoten
- Durchflussraten und Druckverluste
- Wärmeverluste pro Rohrleitung
- Netzwerk-Topologie-Visualisierung
- Vergleich verschiedener Szenarien

---

## 1. Setup

In [ ]:
# Minimal-Bootstrap
import sys
from pathlib import Path

# Find project root
current = Path.cwd()
for candidate in [current] + list(current.parents):
    if (candidate / 'energis').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        PROJECT_ROOT = candidate
        break

print(f"✅ Project root: {PROJECT_ROOT}")

In [ ]:
# Imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yaml
from datetime import datetime

print("✅ Imports erfolgreich")

## 2. Load Network Results

Lade die Ergebnisse einer Optimierung mit aktiviertem thermischen Netzwerk.

In [ ]:
# Wähle Export-Verzeichnis
# Beispiel: exports/20251210_153045_stadtbach-1week-thermal-network/
EXPORT_DIR = None  # Automatisch neuestes wählen

if EXPORT_DIR is None:
    # Find latest export with network results
    exports_dir = PROJECT_ROOT / 'exports'
    if exports_dir.exists():
        export_dirs = sorted([d for d in exports_dir.iterdir() if d.is_dir()], reverse=True)
        
        for exp_dir in export_dirs:
            # Check if it has network results
            if (exp_dir / 'pf_network_timeseries.csv').exists():
                EXPORT_DIR = exp_dir
                break
            elif (exp_dir / 'rh_network_timeseries.csv').exists():
                EXPORT_DIR = exp_dir
                break

if EXPORT_DIR is None:
    print("❌ Kein Export mit Netzwerk-Ergebnissen gefunden!")
    print("   Führe zuerst eine Optimierung mit thermal_network.enabled: true aus")
else:
    print(f"✅ Lade Ergebnisse von: {EXPORT_DIR.name}")
    
    # Determine result type
    if (EXPORT_DIR / 'pf_network_timeseries.csv').exists():
        RESULT_TYPE = 'pf'
    elif (EXPORT_DIR / 'rh_network_timeseries.csv').exists():
        RESULT_TYPE = 'rh'
    else:
        RESULT_TYPE = 'mpc'
    
    print(f"   Result type: {RESULT_TYPE.upper()}")

In [ ]:
if EXPORT_DIR:
    # Load network time series
    network_ts_file = EXPORT_DIR / f'{RESULT_TYPE}_network_timeseries.csv'
    network_ts = pd.read_csv(network_ts_file, index_col=0, parse_dates=True)
    
    # Load network summary
    network_summary_file = EXPORT_DIR / f'{RESULT_TYPE}_network_summary.csv'
    network_summary = pd.read_csv(network_summary_file, index_col=0, squeeze=True).to_dict()
    
    print(f"✅ Zeitreihen geladen: {len(network_ts)} Zeitschritte, {len(network_ts.columns)} Variablen")
    print(f"✅ Summary geladen: {len(network_summary)} Metriken")

## 3. Network Summary KPIs

In [ ]:
if EXPORT_DIR:
    print("🌡️  THERMISCHES NETZWERK - ZUSAMMENFASSUNG")
    print("=" * 70)
    
    print(f"\n📏 Netzwerk-Topologie:")
    print(f"  Knoten:              {network_summary.get('Number_of_nodes', 0)}")
    print(f"  Rohrleitungen:       {network_summary.get('Number_of_pipes', 0)}")
    print(f"  Gesamtlänge:         {network_summary.get('Total_pipe_length_m', 0):.0f} m ({network_summary.get('Total_pipe_length_m', 0)/1000:.2f} km)")
    
    print(f"\n🔥 Wärmelieferung:")
    delivered = network_summary.get('Total_heat_delivered_MWh', 0)
    losses = network_summary.get('Total_heat_loss_MWh', 0)
    loss_pct = network_summary.get('Heat_loss_percentage', 0)
    
    print(f"  Geliefert:           {delivered:.1f} MWh")
    print(f"  Verluste:            {losses:.1f} MWh")
    print(f"  Verlustrate:         {loss_pct:.2f}%")
    
    # Specific losses per km
    if network_summary.get('Total_pipe_length_m', 0) > 0:
        losses_per_km = losses / (network_summary.get('Total_pipe_length_m', 0) / 1000)
        print(f"  Spez. Verluste:      {losses_per_km:.2f} MWh/km")
    
    # Efficiency rating
    print(f"\n📊 Bewertung:")
    if loss_pct < 0.5:
        print(f"  ⭐⭐⭐ Ausgezeichnet! Sehr geringe Verluste für ein Fernwärmenetz")
    elif 0.5 <= loss_pct <= 1.5:
        print(f"  ✅ Gut - Verluste im typischen Bereich moderner Fernwärmenetze")
    elif 1.5 < loss_pct <= 3.0:
        print(f"  ⚠️  Erhöhte Verluste - Isolierung oder Rohrdurchmesser prüfen")
    else:
        print(f"  🔴 Hohe Verluste - Dringend Optimierungsbedarf!")
    
    # Cost estimation (rough)
    heat_price_eur_mwh = 80  # Typical heat generation cost
    loss_cost_eur = losses * heat_price_eur_mwh
    print(f"\n💰 Verlustkosten (geschätzt):")
    print(f"  Bei {heat_price_eur_mwh} EUR/MWh: {loss_cost_eur:.0f} EUR für Simulationszeitraum")

## 4. Temperature Profiles

Visualisiere Vorlauf- und Rücklauftemperaturen an allen Knoten.

In [ ]:
if EXPORT_DIR:
    # Get all temperature columns
    temp_cols = [col for col in network_ts.columns if '_T_supply_C' in col or '_T_return_C' in col]
    
    if temp_cols:
        # Create figure
        fig = go.Figure()
        
        # Plot supply temperatures
        for col in temp_cols:
            if '_T_supply_C' in col:
                node_id = col.replace('NET_', '').replace('_T_supply_C', '')
                fig.add_trace(go.Scatter(
                    x=network_ts.index,
                    y=network_ts[col],
                    name=f"{node_id} (VL)",
                    mode='lines',
                    line=dict(width=2)
                ))
        
        # Plot return temperatures (dashed)
        for col in temp_cols:
            if '_T_return_C' in col:
                node_id = col.replace('NET_', '').replace('_T_return_C', '')
                fig.add_trace(go.Scatter(
                    x=network_ts.index,
                    y=network_ts[col],
                    name=f"{node_id} (RL)",
                    mode='lines',
                    line=dict(width=1, dash='dash')
                ))
        
        fig.update_layout(
            title="Temperaturverläufe im Netzwerk",
            xaxis_title="Zeit",
            yaxis_title="Temperatur [°C]",
            height=600,
            hovermode='x unified'
        )
        
        fig.show()
        
        print(f"\n✅ {len([c for c in temp_cols if '_T_supply_C' in c])} Knoten visualisiert")
    else:
        print("⚠️  Keine Temperatur-Daten gefunden")

## 5. Heat Losses Analysis

In [ ]:
if EXPORT_DIR:
    # Get heat loss columns
    loss_supply_cols = [col for col in network_ts.columns if '_Q_loss_supply_kW' in col]
    loss_return_cols = [col for col in network_ts.columns if '_Q_loss_return_kW' in col]
    
    if loss_supply_cols:
        fig = make_subplots(
            rows=2, cols=1,
            subplot_titles=('Vorlauf-Verluste', 'Rücklauf-Verluste'),
            vertical_spacing=0.12
        )
        
        # Supply losses
        for col in loss_supply_cols:
            pipe_id = col.replace('NET_', '').replace('_Q_loss_supply_kW', '')
            fig.add_trace(
                go.Scatter(x=network_ts.index, y=network_ts[col], name=pipe_id, mode='lines'),
                row=1, col=1
            )
        
        # Return losses
        for col in loss_return_cols:
            pipe_id = col.replace('NET_', '').replace('_Q_loss_return_kW', '')
            fig.add_trace(
                go.Scatter(x=network_ts.index, y=network_ts[col], name=pipe_id, mode='lines', showlegend=False),
                row=2, col=1
            )
        
        fig.update_xaxes(title_text="Zeit", row=2, col=1)
        fig.update_yaxes(title_text="Verluste [kW]", row=1, col=1)
        fig.update_yaxes(title_text="Verluste [kW]", row=2, col=1)
        fig.update_layout(height=800, title_text="Wärmeverluste pro Rohrleitung")
        
        fig.show()
        
        # Calculate total losses per pipe
        print("\n📉 Gesamtverluste pro Rohrleitung:")
        for supply_col, return_col in zip(loss_supply_cols, loss_return_cols):
            pipe_id = supply_col.replace('NET_', '').replace('_Q_loss_supply_kW', '')
            total_supply = network_ts[supply_col].mean()
            total_return = network_ts[return_col].mean()
            total_loss = total_supply + total_return
            print(f"  {pipe_id:30s}: {total_loss:6.2f} kW (VL: {total_supply:.2f}, RL: {total_return:.2f})")
    else:
        print("⚠️  Keine Verlust-Daten gefunden")

## 6. Flow Rates

In [ ]:
if EXPORT_DIR:
    # Get flow columns
    flow_cols = [col for col in network_ts.columns if '_flow_kg_s' in col]
    
    if flow_cols:
        fig = go.Figure()
        
        for col in flow_cols:
            pipe_id = col.replace('NET_', '').replace('_flow_kg_s', '')
            fig.add_trace(go.Scatter(
                x=network_ts.index,
                y=network_ts[col],
                name=pipe_id,
                mode='lines'
            ))
        
        fig.update_layout(
            title="Massenströme im Netzwerk",
            xaxis_title="Zeit",
            yaxis_title="Massenstrom [kg/s]",
            height=500,
            hovermode='x unified'
        )
        
        fig.show()
        
        # Statistics
        print("\n💧 Durchfluss-Statistiken:")
        for col in flow_cols:
            pipe_id = col.replace('NET_', '').replace('_flow_kg_s', '')
            mean_flow = network_ts[col].mean()
            max_flow = network_ts[col].max()
            print(f"  {pipe_id:30s}: Ø {mean_flow:6.2f} kg/s, max {max_flow:6.2f} kg/s")
    else:
        print("⚠️  Keine Durchfluss-Daten gefunden")

## 7. Export Analysis Report

In [ ]:
if EXPORT_DIR:
    # Create detailed analysis report
    report_file = EXPORT_DIR / 'thermal_network_analysis_report.txt'
    
    with open(report_file, 'w', encoding='utf-8') as f:
        f.write("THERMAL NETWORK ANALYSIS REPORT\n")
        f.write("=" * 70 + "\n\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Export: {EXPORT_DIR.name}\n")
        f.write(f"Result Type: {RESULT_TYPE.upper()}\n\n")
        
        f.write("SUMMARY\n")
        f.write("-" * 70 + "\n")
        for key, value in network_summary.items():
            f.write(f"{key:40s}: {value}\n")
        
        f.write("\n" + "=" * 70 + "\n")
    
    print(f"✅ Analyse-Report gespeichert: {report_file.name}")

---

## 📚 Weitere Analysen

Für erweiterte Analysen siehe:
- `docs/DASHBOARD_PREPARATION.md` - Dashboard-Visualisierung
- `docs/STADTBACH_REAL_DATA_OPTIMIZATION.md` - Reale Daten nutzen
- `docs/PERFORMANCE_OPTIMIZATION_THERMAL_NETWORKS.md` - Performance-Tuning